In [ ]:
import sys; sys.path.insert(0, "..")
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML
import src.KZ_2D as kz2

In [ ]:
TAUS     = [5,20,40]      # quench time
N       = 512     # sites per side
NFRAMES = 200     # frames in the animation
SEED    = 88

In [ ]:
import sys; sys.path.insert(0, "..")
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display
import src.KZ_2D as kz2

kz2.dx, kz2.eta, kz2.theta = 0.5, 1.0, 1e-8
kz2.EPS_I, kz2.EPS_F = -1.5, 1.0
kz2.N = N

runs = {}
for TAU in TAUS:
    kz2.TAU = float(TAU)
    t_hist, dt = kz2.make_time_grid(n_steps=int(round(25 * TAU)))
    t0, t1 = t_hist[0], t_hist[-1]
    stride = max(1, (len(t_hist) - 1) // NFRAMES)

    rng = np.random.default_rng(SEED)
    phi = np.zeros((1, N, N))
    pi  = np.zeros((1, N, N))
    amp = np.sqrt(2 * kz2.eta * kz2.theta * dt / kz2.dx**2)

    frames, times = [], []
    for step in range(1, len(t_hist)):
        phi, pi = kz2.rk4_step(phi, pi, t_hist[step - 1], dt, t0, t1)
        pi += amp * rng.standard_normal(phi.shape)
        if step % stride == 0 or step == len(t_hist) - 1:
            frames.append(phi[0].astype(np.float32).copy())
            times.append(t_hist[step])

    runs[TAU] = (np.array(frames), np.array(times))
    print(f"tau={TAU}: {len(frames)} frames")

In [ ]:
plt.rcParams["animation.embed_limit"] = 200   # MB

for TAU in TAUS:
    frames, times = runs[TAU]

    fig, ax = plt.subplots(1, 2, figsize=(14, 7), dpi=80)
    im0 = ax[0].imshow(frames[0], cmap="RdBu_r", vmin=-1, vmax=1, interpolation="nearest")
    im1 = ax[1].imshow(frames[0] > 0, cmap="gray", vmin=0, vmax=1, interpolation="nearest")
    ax[0].set_title(r"$\phi$  (rescaled each frame)", fontsize=16)
    ax[1].set_title(r"sign$(\phi)$", fontsize=16)
    for a in ax:
        a.set_xticks([]); a.set_yticks([])
    title = fig.suptitle("", fontsize=20)

    def update(k, frames=frames, times=times, TAU=TAU, im0=im0, im1=im1, title=title):
        f = frames[k]
        v = 2.5 * f.std() + 1e-30
        im0.set_data(f); im0.set_clim(-v, v)
        im1.set_data(f > 0)
        title.set_text(f"$\\tau_Q$ = {TAU}      $t/\\tau_Q$ = {times[k]/TAU:+.2f}")
        return im0, im1, title

    anim = animation.FuncAnimation(fig, update, frames=len(frames), interval=50)
    plt.close(fig)
    display(HTML(anim.to_jshtml()))

In [1]:
import sys; sys.path.insert(0, "..")
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
import src.kz_2d as kz2

In [2]:
# ---- inputs ----
TAUS     = [5, 20, 40]    # quench times (one panel each)
N        = 512            # sites per side
NFRAMES  = 120            # frames in the GIF
SEED     = 88
FPS      = 20
OUTFILE  = "../figures/2d/kz_quench.gif"

kz2.dx, kz2.eta, kz2.theta = 0.5, 1.0, 1e-8
kz2.EPS_I, kz2.EPS_F = -1.5, 1.0
kz2.N = N

In [3]:
# ---- simulate: one run per tau, keep NFRAMES evenly spaced snapshots ----
# custom loop instead of kz2.run(): run() also stores pi and every stride-th step, too much memory at N=512
runs = {}
for TAU in TAUS:
    kz2.TAU = float(TAU)
    t_hist, dt = kz2.make_time_grid(n_steps=int(round(25 * TAU)))   # dt = 0.1
    t0, t1 = t_hist[0], t_hist[-1]
    stride = max(1, (len(t_hist) - 1) // NFRAMES)

    rng = np.random.default_rng(SEED)
    phi = np.zeros((N, N))
    pi  = np.zeros((N, N))
    amp = np.sqrt(2 * kz2.eta * kz2.theta * dt / kz2.dx**2)

    frames, times = [], []
    for step in range(1, len(t_hist)):
        phi, pi = kz2.rk4_step(phi, pi, t_hist[step - 1], dt, t0, t1)
        pi += amp * rng.standard_normal(phi.shape)
        if step % stride == 0:
            frames.append(phi.astype(np.float32).copy())
            times.append(t_hist[step])

    runs[TAU] = (np.array(frames[:NFRAMES]), np.array(times[:NFRAMES]))
    print(f"tau={TAU}: {len(runs[TAU][0])} frames")

tau=5: 120 frames
tau=20: 120 frames
tau=40: 120 frames


In [8]:
# ---- animate: one panel per tau, same t/tau in every panel ----
fig, axes = plt.subplots(1, len(TAUS), figsize=(5 * len(TAUS), 5.4), dpi=55)
ims = []
for ax, TAU in zip(axes, TAUS):
    im = ax.imshow(runs[TAU][0][0], cmap="RdBu_r", vmin=-1, vmax=1, interpolation="nearest")
    ax.set_title(rf"$\tau_Q = {TAU}$", fontsize=18)
    ax.set_xticks([]); ax.set_yticks([])
    ims.append(im)
title = fig.suptitle("", fontsize=20)

def update(k):
    for im, TAU in zip(ims, TAUS):
        f = runs[TAU][0][k]
        v = 2.5 * f.std() + 1e-30          # rescale each frame: field grows by orders of magnitude
        im.set_data(f); im.set_clim(-v, v)
    t_frac = runs[TAUS[0]][1][k] / TAUS[0]
    title.set_text(rf"$t/\tau_Q = {t_frac:+.2f}$")
    return ims + [title]
frame_order = list(range(NFRAMES)) + [NFRAMES - 1] * 120
anim = animation.FuncAnimation(fig, update, frames=frame_order, interval=1000 / FPS)
#anim = animation.FuncAnimation(fig, update, frames=NFRAMES, interval=1000 / FPS)
os.makedirs(os.path.dirname(OUTFILE), exist_ok=True)
anim.save(OUTFILE, writer="pillow", fps=FPS)
plt.close(fig)
print("saved:", OUTFILE, f"({os.path.getsize(OUTFILE) / 1e6:.1f} MB)")

saved: ../figures/2d/kz_quench.gif (17.2 MB)
